# CardioSense — Guideline-Grounded Retrieval (RAG)

Turns an assessment into a clinician-facing report in which **every clinical
statement is either a model output or a verbatim guideline quotation with a page
number**.

---

## The design decision that matters most

The default generator is **extractive**. It copies guideline sentences verbatim
with their citations and composes nothing.

It therefore **cannot** hallucinate — not "is unlikely to", *cannot*, because no
text is generated. An LLM would write fluent prose and occasionally invent a
recommendation that no guideline contains, and in a clinical report that failure
is invisible until someone checks the source.

A large language model can be added behind a flag, but then its hallucination
rate has to be **measured** rather than assumed, which is a different project.

## Where hallucination would actually enter

Not at generation. **At query construction.**

If a query could invent *"chest pain on exertion"* because the risk score came
out high, retrieval would faithfully fetch evidence for a patient who does not
exist, and every downstream step would be correctly grounded in that fiction.

So `build_queries()` reads model findings and nothing else, and each query
records the finding that produced it.

## Three content types, always visibly separate

| label | meaning |
|---|---|
| `AI MODEL FINDING` | what a CardioSense model estimated |
| `GUIDELINE EVIDENCE` | text reproduced verbatim, with citation |
| `GENERATED SUMMARY` | narrative composed by the system |

Conflating them is the central safety risk here. A clinician who cannot tell a
model's guess from a guideline's recommendation has lost the ability to weigh
either.

## On the PDFs

**None are redistributed with this project.** Guidelines are free to read but
carry their own redistribution terms, so the repository ships a manifest of
pointers and you download them yourself. One document is enough to start.

## 1. Environment

In [17]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = Path("/content/CardioSense")
DRIVE = Path("/content/drive/MyDrive/CardioSense")
import sys
from pathlib import Path

REPO_DIR = Path("/content/drive/MyDrive/CardioSense")
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Source directory added:", SRC_DIR)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    os.chdir(REPO_DIR)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)],check=True)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q","sentence-transformers", "pypdf"],check=True)

else:
    here = Path.cwd()
    while not (here / "pyproject.toml").exists() and here != here.parent:
        here = here.parent
    os.chdir(here); REPO_DIR = here

src = str(REPO_DIR / "src")
if src not in sys.path:
    sys.path.insert(0, src)

KB = (DRIVE if IN_COLAB else REPO_DIR) / "knowledge_base"
GUIDELINES = KB / "guidelines"
INDEX_DIR = KB / "index"
RESULTS = ((DRIVE if IN_COLAB else REPO_DIR) / "results") / "rag"
for d in (GUIDELINES, INDEX_DIR, RESULTS):
    d.mkdir(parents=True, exist_ok=True)

print("knowledge base :", KB)
print("guidelines     :", GUIDELINES)
print("index          :", INDEX_DIR)

Source directory added: /content/drive/MyDrive/CardioSense/src
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
knowledge base : /content/drive/MyDrive/CardioSense/knowledge_base
guidelines     : /content/drive/MyDrive/CardioSense/knowledge_base/guidelines
index          : /content/drive/MyDrive/CardioSense/knowledge_base/index


## 2. The manifest — which documents to download

This writes `guidelines/manifest.json` listing each registered document with its
URL. Download the PDFs yourself into that folder.

**Verify every title and year against the issuing body before citing anything.**
Guidelines are revised, and a citation to a superseded edition is worse than no
citation at all.

In [18]:
import json
from cardiosense.rag import REGISTRY, check_available, write_manifest

manifest_path = write_manifest(GUIDELINES / "manifest.json")
print("wrote", manifest_path, "\n")

print(f"{'doc_id':<36}{'org':<12}{'year':<7}status")
print("-" * 76)
availability = check_available(GUIDELINES)
for doc in REGISTRY:
    mark = "PRESENT" if doc.doc_id in availability["present"] else "missing"
    org = doc.organization.split()[0][:10]
    print(f"{doc.doc_id:<36}{org:<12}{doc.year:<7}{mark}")

print(f"\n{availability['n_present']} of {len(REGISTRY)} present")
if not availability["ready"]:
    print("\nDownload at least one PDF into:", GUIDELINES)
    print("Start with the ESC prevention guideline. The notebook runs either way —")
    print("without an index, every report states that nothing in it is")
    print("guideline-derived, which is the honest behaviour rather than an error.")
for doc in REGISTRY:
    if doc.doc_id in availability["missing"]:
        print(f"\n  {doc.filename}\n    {doc.url}")

wrote /content/drive/MyDrive/CardioSense/knowledge_base/guidelines/manifest.json 

doc_id                              org         year   status
----------------------------------------------------------------------------
esc_cvd_prevention_2021             European    2021   PRESENT
esc_acs_2023                        European    2023   PRESENT
esc_hf_2021                         European    2021   PRESENT
acc_aha_primary_prevention_2019     American    2019   PRESENT
who_hearts_2020                     World       2020   PRESENT

5 of 5 present


## 3. Chunking, demonstrated on a worked example

Chunks are **1000 characters with 200 of overlap**, split on paragraph then
sentence boundaries, and **never across a page break**.

* 1000 characters is roughly a paragraph — the unit a recommendation is actually
  written in — and it fits inside the embedding model's 256-token window, so
  nothing is silently truncated.
* The overlap stops a recommendation that straddles a boundary from being
  retrievable in neither chunk.
* One page per chunk means every citation can be checked by opening that page.

Running headers are stripped first. Left in, they dominate the embedding of
every short chunk, and a search for anything retrieves the journal name.

In [19]:
from cardiosense.rag import chunk_document, clean_text

demo_pages = [
    "European Heart Journal 2021\n\n"
    "Blood pressure should be lowered to below 140/90 mmHg in all treated "
    "patients. A target of below 130/80 mmHg should be considered where "
    "treatment is well tolerated.\n\n"
    + "Risk factor control remains central to cardiovascular prevention. " * 10,

    "European Heart Journal 2021\n\n"
    "In patients with suspected myocardial ischaemia, ST segment and T wave "
    "changes on the resting electrocardiogram warrant further evaluation.\n\n"
    + "Exercise testing may be considered for risk stratification. " * 10,

    "European Heart Journal 2021\n\n"
    "An enlarged cardiac silhouette on chest radiography may indicate "
    "cardiomegaly and should prompt echocardiographic assessment of "
    "ventricular function.\n\n"
    + "Echocardiography remains the reference investigation. " * 10,
]

demo_chunks = chunk_document(demo_pages, "demo_guideline",
                             organization="European Society of Cardiology",
                             year=2021, title="Demonstration guideline")

print(f"{len(demo_pages)} pages -> {len(demo_chunks)} chunks\n")
header_gone = not any("European Heart Journal 2021" in c.text for c in demo_chunks)
print(f"  running header stripped        : {header_gone}")
print(f"  one page number per chunk      : {all(c.page_number for c in demo_chunks)}")
print(f"  all within the size limit      : {max(len(c.text) for c in demo_chunks) <= 1000}")
print(f"\nexample chunk:\n  id       : {demo_chunks[0].chunk_id}")
print(f"  citation : {demo_chunks[0].citation}")
print(f"  text     : {demo_chunks[0].text[:150]}...")

3 pages -> 3 chunks

  running header stripped        : True
  one page number per chunk      : True
  all within the size limit      : True

example chunk:
  id       : demo_guideline:p1:c0
  citation : European Society of Cardiology, 2021, p. 1
  text     : Blood pressure should be lowered to below 140/90 mmHg in all treated patients. A target of below 130/80 mmHg should be considered where treatment is w...


## 4. Build the index

**Exact search, not approximate.** A few thousand chunks scan in under a
millisecond. FAISS's approximate indexes trade recall for speed on *millions* of
vectors — choosing one here would mean a guideline passage could be **silently
missed**, in a system whose whole purpose is grounding claims in retrievable
text.

Embeddings come from `all-MiniLM-L6-v2` (~80 MB, downloaded once).

Skipped automatically if no PDFs are present.

In [20]:
from cardiosense.rag import build_knowledge_base

if availability["ready"]:
    build_report = build_knowledge_base(GUIDELINES, INDEX_DIR)
    print(json.dumps(build_report, indent=2)[:900])
else:
    build_report = {"built": False, "reason": "no PDFs present"}
    print("No guideline PDFs found — skipping the index build.")
    print("Everything below still runs; retrieval simply returns nothing and")
    print("the report says so explicitly.")

  esc_cvd_prevention_2021             111 pages ->  868 chunks
  esc_acs_2023                        107 pages ->  843 chunks
  esc_hf_2021                         128 pages ->  996 chunks
  acc_aha_primary_prevention_2019      51 pages ->  340 chunks
  who_hearts_2020                      80 pages ->  267 chunks

  embedding 3314 chunks with all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/content/drive/MyDrive/CardioSense/src/cardiosense/rag/index.py:47: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dimension = int(self._model.get_sentence_embedding_dimension())


Batches:   0%|          | 0/104 [00:00<?, ?it/s]

{
  "built": true,
  "n_chunks": 3314,
  "documents": [
    {
      "doc_id": "esc_cvd_prevention_2021",
      "pages": 111,
      "chunks": 868
    },
    {
      "doc_id": "esc_acs_2023",
      "pages": 107,
      "chunks": 843
    },
    {
      "doc_id": "esc_hf_2021",
      "pages": 128,
      "chunks": 996
    },
    {
      "doc_id": "acc_aha_primary_prevention_2019",
      "pages": 51,
      "chunks": 340
    },
    {
      "doc_id": "who_hearts_2020",
      "pages": 80,
      "chunks": 267
    }
  ],
  "missing_documents": [],
  "embedding_model": "all-MiniLM-L6-v2",
  "index_dir": "/content/drive/MyDrive/CardioSense/knowledge_base/index"
}


### Two refusals worth knowing about

The index refuses to be built or loaded in two situations where it would
otherwise fail *silently* — which is far worse than failing loudly.

In [21]:
from cardiosense.rag import HashingBackend, build_index

print("1. A non-semantic backend is refused for production:")
try:
    build_index(demo_chunks, HashingBackend())
    print("   FAIL — it was accepted")
except ValueError as exc:
    print(f"   refused: {str(exc)[:150]}...")

print("\n2. A query embedded by a different model than the index lands in a")
print("   different vector space. Every score is then meaningless, but nothing")
print("   crashes — so VectorIndex.load() compares the model name and refuses.")

demo_index = build_index(demo_chunks, HashingBackend(), allow_non_semantic=True)
print(f"\n   (test index built with the offline backend: {len(demo_index)} chunks)")

1. A non-semantic backend is refused for production:
   refused: 'hashing-offline' is not a semantic embedding model. It matches on shared words, not meaning, so retrieval will look like it works and be wrong. Use S...

2. A query embedded by a different model than the index lands in a
   different vector space. Every score is then meaningless, but nothing
   crashes — so VectorIndex.load() compares the model name and refuses.

   (test index built with the offline backend: 3 chunks)


## 5. Queries are derived from findings, never invented

This is the safety-critical step. Each query records the finding that produced
it, so a retrieved passage can always be traced back to the model output that
asked for it.

In [22]:
from cardiosense.fusion import FusionEngine, ModalityPrediction
from cardiosense.rag import build_queries

maths = FusionEngine(adapters={})
#risk score for demonstration purpose. input real score in backend file
assessment = maths.fuse_predictions([
    ModalityPrediction(modality="clinical", risk_score=0.86, reliability=0.92,
                       is_calibrated=True, native_labels={"coronary_stenosis": 1}),
    ModalityPrediction(modality="ecg", risk_score=0.74, reliability=0.50,
                       is_calibrated=False,
                       native_labels={"NORM": 0, "MI": 0, "STTC": 1, "CD": 0, "HYP": 1}),
    ModalityPrediction(modality="xray", risk_score=0.79, reliability=0.88,
                       is_calibrated=True, native_labels={"cardiomegaly": 1}),
]).to_dict()

queries = build_queries(assessment)
print(f"{len(queries)} queries, each traceable to a finding:\n")
for q in queries:
    print(f"  [{q.topic:<18}] {q.text[:44]:<44}")
    print(f"   {'':<20} <- {q.source_finding}")

assert all(q.source_finding for q in queries)
print("\nOK: every query traces to a model finding. None was invented.")

print("\nThe PTB-XL codes are translated into guideline vocabulary:")
print("  STTC -> 'ST segment and T wave changes, myocardial ischaemia'")
print("  HYP  -> 'left ventricular hypertrophy'")
print("This is the dataset's own documented expansion of its codes, not")
print("new clinical information being added.")

5 queries, each traceable to a finding:

  [overall_risk      ] high cardiovascular risk management, intensi
                        <- fused risk level = HIGH
  [clinical_cad      ] obstructive coronary artery disease, chronic
                        <- clinical model predicted coronary stenosis present
  [ecg_sttc          ] ST segment and T wave changes, myocardial is
                        <- ECG model predicted STTC present
  [ecg_hyp           ] left ventricular hypertrophy                
                        <- ECG model predicted HYP present
  [xray_cardiomegaly ] cardiomegaly, enlarged cardiac silhouette, c
                        <- X-ray model predicted cardiomegaly present

OK: every query traces to a model finding. None was invented.

The PTB-XL codes are translated into guideline vocabulary:
  STTC -> 'ST segment and T wave changes, myocardial ischaemia'
  HYP  -> 'left ventricular hypertrophy'
This is the dataset's own documented expansion of its codes, not
new clin

## 6. Retrieval

In [23]:
from cardiosense.rag import RAGService

service = RAGService(INDEX_DIR if build_report.get("built") else None)
print(json.dumps(service.status(), indent=2))

if service.retrieval_available:
    print("\n--- example searches " + "-" * 50)
    for probe in ("blood pressure target in treated hypertension",
                  "cardiomegaly on chest radiography",
                  "ST segment changes and myocardial ischaemia"):
        print(f"\nquery: {probe}")
        for hit in service.search(probe, k=2):
            print(f"  {hit.score:.3f}  {hit.chunk.citation}")
            print(f"         {hit.chunk.text[:110]}...")
else:
    print("\nRetrieval unavailable — build the index to enable it.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{
  "retrieval_available": true,
  "n_chunks": 3314,
  "embedding_model": "all-MiniLM-L6-v2",
  "provider": "extractive",
  "can_hallucinate": false,
  "load_error": "",
  "hint": ""
}

--- example searches --------------------------------------------------

query: blood pressure target in treated hypertension
  0.676  European Society of Cardiology, 2021, p. 59
         such as
renal disease, haemorrhagic cerebrovascular disease, and HF. 4.7.5.3 Blood pressure treatment targets
...
  0.654  European Society of Cardiology, 2021, p. 106
         Cushman WC, Cheung AK, Ambrosius WT. A randomized trial of intensive versus standard blood-pressure control. N...

query: cardiomegaly on chest radiography
  0.569  European Society of Cardiology, 2021, p. 98
         inagreement with recent chamber quantification, diastolic function, and heart valvedisease recommendations: an...
  0.537  American College of Cardiology / American Heart Association, 2019, p. 51
         10, 2019 Circulation. 2019

/content/drive/MyDrive/CardioSense/src/cardiosense/rag/index.py:47: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dimension = int(self._model.get_sentence_embedding_dimension())


In [24]:
import re

retrieval_tests = [
    {
        "name": "Cardiomegaly on chest radiography",
        "query": "cardiomegaly enlarged cardiac silhouette on chest X-ray",
        "expected": ["cardiomegaly", "cardiac silhouette", "chest radiograph"],
    },
    {
        "name": "ECG ST-T abnormalities",
        "query": "ST segment depression T wave inversion ECG myocardial ischaemia",
        "expected": ["st-segment", "t wave", "ischaemia", "ischemia"],
    },
    {
        "name": "Left ventricular hypertrophy",
        "query": "left ventricular hypertrophy ECG evidence and evaluation",
        "expected": ["ventricular hypertrophy", "left ventricular", "hypertrophy"],
    },
    {
        "name": "Blood pressure targets",
        "query": "blood pressure treatment targets in treated hypertension",
        "expected": ["blood pressure", "mmhg", "hypertension"],
    },
]

assert service.retrieval_available, "RAG retrieval is unavailable. Build/load the index first."

for test in retrieval_tests:
    print("\n" + "=" * 85)
    print("TEST:", test["name"])
    print("QUERY:", test["query"])

    hits = service.search(test["query"], k=5)
    if not hits:
        print("NO RESULTS — investigate the index or retrieval pipeline.")
        continue

    for rank, hit in enumerate(hits, start=1):
        chunk_text = hit.chunk.text
        text_lower = chunk_text.lower()
        matched_terms = [
            term for term in test["expected"] if term in text_lower
        ]

        print(f"\n{rank}. Score: {hit.score:.3f}")
        print("Citation:", hit.chunk.citation)
        print("Expected terms found:", matched_terms or "NONE")
        print("Passage:", re.sub(r"\s+", " ", chunk_text)[:500])

        if not matched_terms:
            print("REVIEW: this result may be weakly relevant.")


TEST: Cardiomegaly on chest radiography
QUERY: cardiomegaly enlarged cardiac silhouette on chest X-ray

1. Score: 0.461
Citation: European Society of Cardiology, 2021, p. 98
Expected terms found: NONE
Passage: inagreement with recent chamber quantification, diastolic function, and heart valvedisease recommendations: an expert consensus document of the EuropeanAssociation of Cardiovascular Imaging. Eur Heart J Cardiovasc Imaging2017;18:1301/C01310.17. Arrigo M, Huber LC, Winnik S, Mikulicic F, Guidetti F, Frank M, Flammer AJ,Ruschitzka F. Right ventricular failure: pathophysiology, diagnosis and treatment.Card Fail Rev 2019;5:140/C0146.18. Gorter TM, van Veldhuisen DJ, Bauersachs J, Borlaug BA, Celutki
REVIEW: this result may be weakly relevant.

2. Score: 0.458
Citation: European Society of Cardiology, 2023, p. 83
Expected terms found: NONE
Passage: Society of Cardiology, Toomas Marandi; Finland: Finnish Cardiac Society, Essi Ryödi; France: French Society of Cardiology, Jean-Philippe 

In [25]:
import re
from pathlib import Path
from pypdf import PdfReader
from cardiosense.rag import REGISTRY

def normalize_text(value):
    return re.sub(r"\s+", " ", value or "").strip().lower()

# Build a lookup from registered document ID to its local PDF.
pdf_paths = {
    doc.doc_id: GUIDELINES / doc.filename
    for doc in REGISTRY
}

# Collect results from the evaluation queries above.
citation_hits = []
for test in retrieval_tests:
    for hit in service.search(test["query"], k=5):
        citation_hits.append(hit)

# Cache extracted PDF pages to avoid repeatedly opening the same file.
pdf_page_cache = {}
citation_audit = []

for hit in citation_hits:
    chunk = hit.chunk
    doc_id = getattr(chunk, "doc_id", None)
    page_number = getattr(chunk, "page_number", None)
    citation = getattr(chunk, "citation", "")
    chunk_text = normalize_text(getattr(chunk, "text", ""))

    pdf_path = pdf_paths.get(doc_id)
    status = "REVIEW"
    detail = ""

    if not pdf_path or not pdf_path.exists():
        detail = f"Source PDF not found for doc_id={doc_id!r}"
    elif not page_number:
        detail = "Chunk has no page number"
    else:
        try:
            if doc_id not in pdf_page_cache:
                reader = PdfReader(str(pdf_path))
                pdf_page_cache[doc_id] = [
                    normalize_text(page.extract_text() or "")
                    for page in reader.pages
                ]

            pages = pdf_page_cache[doc_id]

            if page_number < 1 or page_number > len(pages):
                detail = f"Page {page_number} is outside PDF page range 1-{len(pages)}"
            else:
                source_page = pages[page_number - 1]

                if chunk_text and chunk_text in source_page:
                    status = "MATCH"
                    detail = "Chunk text found on recorded PDF page"
                else:
                    # Cleaning/line-break differences can prevent an exact match.
                    chunk_words = set(chunk_text.split())
                    page_words = set(source_page.split())
                    overlap = (
                        len(chunk_words & page_words) / len(chunk_words)
                        if chunk_words else 0
                    )
                    status = "REVIEW"
                    detail = f"Exact match failed; word overlap={overlap:.0%}"

        except Exception as exc:
            detail = f"Could not inspect PDF: {exc!r}"

    citation_audit.append({
        "status": status,
        "doc_id": doc_id,
        "page_number": page_number,
        "citation": citation,
        "detail": detail,
    })

for item in citation_audit:
    print(
        f"[{item['status']}] {item['citation']} | "
        f"{item['detail']}"
    )

matches = sum(x["status"] == "MATCH" for x in citation_audit)
reviews = sum(x["status"] != "MATCH" for x in citation_audit)

print(f"\nAudit summary: {matches} matched; {reviews} need review.")
print("A REVIEW result is not automatically a wrong citation; inspect that PDF page.")

[MATCH] European Society of Cardiology, 2021, p. 98 | Chunk text found on recorded PDF page
[REVIEW] European Society of Cardiology, 2023, p. 83 | Exact match failed; word overlap=97%
[MATCH] European Society of Cardiology, 2021, p. 82 | Chunk text found on recorded PDF page
[REVIEW] European Society of Cardiology, 2023, p. 97 | Exact match failed; word overlap=99%
[REVIEW] European Society of Cardiology, 2021, p. 20 | Exact match failed; word overlap=97%
[REVIEW] European Society of Cardiology, 2023, p. 19 | Exact match failed; word overlap=99%
[REVIEW] European Society of Cardiology, 2023, p. 19 | Exact match failed; word overlap=99%
[MATCH] European Society of Cardiology, 2023, p. 31 | Chunk text found on recorded PDF page
[REVIEW] European Society of Cardiology, 2023, p. 19 | Exact match failed; word overlap=97%
[REVIEW] European Society of Cardiology, 2023, p. 29 | Exact match failed; word overlap=98%
[MATCH] European Society of Cardiology, 2021, p. 82 | Chunk text found on record

## 7. The grounded report

Assembled from the stored assessment, never regenerated from scratch, so it
cannot state anything the pipeline did not produce.

In [26]:
report, used_queries = service.generate_report(assessment)

print(f"provider        : {report.provider}")
print(f"can hallucinate : {report.can_hallucinate}")
print(f"sections        : {len(report.sections)}")
print(f"citations       : {len(report.citations)}\n")

print(f"{'section':<36}{'content type':<22}cites")
print("-" * 68)
for s in report.sections:
    print(f"{s.heading:<36}{s.content_type:<22}{len(s.citations)}")

types = {s.content_type for s in report.sections}
assert len(types) == 3, "all three content types must be present and separate"
print(f"\nOK: {sorted(types)} kept visibly separate")

if report.warnings:
    print("\nwarnings:")
    for w in report.warnings:
        print("  -", w)

provider        : extractive
can hallucinate : False
sections        : 6
citations       : 6

section                             content type          cites
--------------------------------------------------------------------
Model Findings                      AI MODEL FINDING      0
Composite Indicator                 AI MODEL FINDING      0
Guideline Evidence                  GUIDELINE EVIDENCE    6
Decision-Support Considerations     GENERATED SUMMARY     0
Limitations and Uncertainty         GENERATED SUMMARY     0
Sources                             GUIDELINE EVIDENCE    6

OK: ['AI MODEL FINDING', 'GENERATED SUMMARY', 'GUIDELINE EVIDENCE'] kept visibly separate


In [27]:
import json
from cardiosense.rag import screen

report_data = report.to_dict()

# Re-run the report's actual queries to build the set of retrieved evidence.
report_hits = []
for query in used_queries:
    hits = service.search(query.text, k=10)
    report_hits.extend(hits)

retrieved_texts = [
    normalize_text(hit.chunk.text)
    for hit in report_hits
]
retrieved_citations = {
    hit.chunk.citation
    for hit in report_hits
}

# Recursively find text/citation records in the guideline-evidence sections.
evidence_records = []

def collect_evidence(obj, in_evidence_section=False):
    if isinstance(obj, dict):
        heading = str(obj.get("heading", "")).lower()
        content_type = str(obj.get("content_type", "")).lower()

        is_evidence = (
            in_evidence_section
            or "guideline evidence" in heading
            or content_type == "guideline evidence"
        )

        if is_evidence and "text" in obj and "citation" in obj:
            evidence_records.append(obj)

        for value in obj.values():
            collect_evidence(value, is_evidence)

    elif isinstance(obj, list):
        for value in obj:
            collect_evidence(value, in_evidence_section)

collect_evidence(report_data)

print("Generator:", report.provider)
print("Can hallucinate:", report.can_hallucinate)
print("Evidence records found:", len(evidence_records))

if not evidence_records:
    print(
        "\nCould not automatically locate structured evidence records. "
        "Inspect report_data's structure below before interpreting this test."
    )
    print(json.dumps(report_data, indent=2, default=str)[:6000])
else:
    grounded_count = 0

    for index, record in enumerate(evidence_records, start=1):
        quote = normalize_text(str(record["text"]))
        citation = str(record["citation"])

        text_match = any(
            quote and (quote in retrieved or retrieved in quote)
            for retrieved in retrieved_texts
        )
        citation_match = citation in retrieved_citations

        grounded = text_match and citation_match
        grounded_count += int(grounded)

        print(f"\nEvidence {index}: {'GROUNDED' if grounded else 'REVIEW'}")
        print("Citation:", citation)
        print("Text found in retrieved evidence:", text_match)
        print("Citation found in retrieved sources:", citation_match)
        print("Passage:", quote[:250])

    print(
        f"\nGrounding check: {grounded_count}/{len(evidence_records)} "
        "evidence records matched retrieved text and citation."
    )

# The existing extractive implementation should remain clearly identified.
assert str(report.provider).lower() == "extractive", (
    "Generator provider differs from the expected extractive implementation."
)

# Run the notebook's existing safety screen against the report.
report_text = report.to_text()
violations = screen(report_text)
print("\nSafety-screen violations:", violations or "None detected")

Generator: extractive
Can hallucinate: False
Evidence records found: 6

Evidence 1: GROUNDED
Citation: European Society of Cardiology, 2023, p. 31
Text found in retrieved evidence: True
Citation found in retrieved sources: True
Passage: shock • recurrent or refractory chest pain despite medical treatment • in-hospital life-threatening arrhythmias • mechanical complications of mi • acute heart failure presumed secondary to ongoing myocardial ischaemia • recurrent dynamic st-segment o

Evidence 2: GROUNDED
Citation: European Society of Cardiology, 2023, p. 19
Text found in retrieved evidence: True
Citation found in retrieved sources: True
Passage: of ongoing myocardial ischaemia, the presence of lbbb, right bundle branch block (rbbb), or a paced rhythm precludes an accurate assessment of the presence or absence of st-segment elevation. therefore, patients presenting with these ecg patterns in 

Evidence 3: GROUNDED
Citation: European Society of Cardiology, 2021, p. 24
Text found in retri

In [28]:
text = report.to_text()
print(text[:2600])

CARDIOSENSE DECISION-SUPPORT REPORT

[AI MODEL FINDING]
Model Findings
--------------
  - clinical: risk 0.860 (calibrated, reliability 0.92); predicted present: coronary_stenosis
  - ecg: risk 0.740 (UNCALIBRATED, reliability 0.50); predicted present: STTC, HYP
  - xray: risk 0.790 (calibrated, reliability 0.88); predicted present: cardiomegaly

[AI MODEL FINDING]
Composite Indicator
-------------------
  composite_risk: 0.819
  risk_level: HIGH
  reliability: 0.836 (HIGH)
  based_on: clinical + ecg + xray
  weights: {'clinical': 0.4689, 'ecg': 0.1699, 'xray': 0.3613}
  note: This is a composite cardiovascular abnormality indicator across models trained on different cohorts for different targets. It is not a diagnosis of any single condition.

[GUIDELINE EVIDENCE]
Guideline Evidence
------------------
  - {'text': 'shock \n• Recurrent or refractory chest pain despite medical \ntreatment \n• In-hospital life-threatening arrhythmias \n• Mechanical complications of MI \n• Acute heart fai

## 8. The safety screen

Every emitted string is screened for language a decision-support prototype must
never use: diagnostic assertion, prescription, drug doses, false certainty.

The screen uses a **200-character prohibition window**, because the two most
awkward false positives are the system's own safety statements:

* *"This report must not be used for prescribing decisions"* contains
  **prescribing** while forbidding it.
* *">120 mg/dl"* is a laboratory unit, not a dose — distinguished by the slash.

A screen that fires on its own disclaimers is worse than no screen. It trains
everyone to ignore the alerts, which is exactly how a real prescriptive sentence
later slips through.

In [29]:
from cardiosense.rag import screen

cases = [
    ("This report must not be used for prescribing decisions.", False,
     "the system's own not-for list"),
    ("CardioSense does not prescribe or recommend treatment.", False,
     "explicit negation"),
    ("Fasting blood sugar >120 mg/dl indicates diabetes.", False,
     "laboratory unit, not a dose"),
    ("Titrate to 10 mg daily as tolerated.", True, "a genuine drug dose"),
    ("The patient has coronary artery disease.", True, "diagnostic assertion"),
    ("This definitely rules out ischaemia.", True, "false certainty"),
    ("No further testing is needed.", True, "discharging a patient"),
]

print(f"{'fires':<8}{'expected':<11}{'verdict':<9}case")
print("-" * 76)
all_ok = True
for sentence, should_fire, why in cases:
    fires = bool(screen(sentence))
    ok = fires == should_fire
    all_ok &= ok
    print(f"{str(fires):<8}{str(should_fire):<11}{'ok' if ok else 'FAIL':<9}{why}")

assert all_ok, "the safety screen misbehaved"
print("\nThe exemptions did not disarm the screen: real doses and real")
print("diagnostic assertions still fire.")

violations = screen(text)
print(f"\nScreen on the full generated report: "
      f"{violations if violations else 'clean'}")

fires   expected   verdict  case
----------------------------------------------------------------------------
False   False      ok       the system's own not-for list
False   False      ok       explicit negation
False   False      ok       laboratory unit, not a dose
True    True       ok       a genuine drug dose
True    True       ok       diagnostic assertion
True    True       ok       false certainty
True    True       ok       discharging a patient

The exemptions did not disarm the screen: real doses and real
diagnostic assertions still fire.

Screen on the full generated report: clean


## 9. Save

In [30]:
import time

(RESULTS / "report_example.txt").write_text(text, encoding="utf-8")
(RESULTS / "report_example.json").write_text(
    json.dumps(report.to_dict(), indent=2), encoding="utf-8")

summary = {
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "knowledge_base": {
        "documents_present": availability["present"],
        "documents_missing": availability["missing"],
        "index_built": bool(build_report.get("built")),
        "n_chunks": build_report.get("n_chunks", 0),
        "embedding_model": build_report.get("embedding_model"),
        "search": "exact inner product",
        "chunk_size": 1000, "chunk_overlap": 200,
    },
    "generation": {
        "provider": report.provider,
        "can_hallucinate": report.can_hallucinate,
        "content_types": sorted({s.content_type for s in report.sections}),
        "n_queries": len(used_queries),
        "queries": [q.to_dict() for q in used_queries],
    },
    "safety_screen": {"report_violations": screen(text)},
    "limitations": [
        "Guideline PDFs are not redistributed; the corpus is only as complete "
        "as what was downloaded.",
        "The extractive generator cannot hallucinate, but it also cannot "
        "synthesise across passages. Retrieval failures become omissions, not "
        "wrong statements.",
        "Retrieval is lexical-semantic over chunk embeddings. A relevant "
        "passage phrased very differently from the query can be missed.",
        "The safety screen is pattern matching with a negation window. It "
        "catches obvious failures and will miss a subtly prescriptive sentence. "
        "It is a floor, not a guarantee.",
        "Guideline editions must be verified against the issuing body before "
        "any citation is relied on.",
    ],
}
(RESULTS / "rag_summary.json").write_text(json.dumps(summary, indent=2),
                                          encoding="utf-8")

for name in ("report_example.txt", "report_example.json", "rag_summary.json"):
    print("saved:", RESULTS / name)

saved: /content/drive/MyDrive/CardioSense/results/rag/report_example.txt
saved: /content/drive/MyDrive/CardioSense/results/rag/report_example.json
saved: /content/drive/MyDrive/CardioSense/results/rag/rag_summary.json


---

## What to say about this in the review

> Guideline passages are retrieved by exact semantic search over 1000-character
> chunks that each carry their organisation, year and page. Queries are built
> only from model findings, each recording the finding that produced it, so no
> patient information is ever invented at retrieval time. Generation is
> extractive — guideline text is copied verbatim with its citation — so the
> report cannot contain a recommendation no guideline makes. Model findings,
> guideline evidence and generated narrative are labelled separately throughout,
> and every emitted string is screened for prescriptive or diagnostic language.

**The strongest line if asked why not an LLM:** the extractive generator cannot
hallucinate by construction. An LLM would need its hallucination rate measured,
not assumed, and that measurement is a project in itself.